In [1]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

DATA_PATH = Path("../olist_data")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("POSTGRES_DB"),
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print(result.fetchone())

('olist_marketplace',)


In [3]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        TRUNCATE TABLE 
            order_reviews,
            order_payments,
            order_items,
            orders,
            customers,
            products,
            sellers,
            geolocation,
            product_category_translation
        RESTART IDENTITY CASCADE;
    """))

print("Tables cleaned successfully.")

Tables cleaned successfully.


In [4]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

DATA_PATH = Path("../olist_data")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("POSTGRES_DB"),
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print(result.fetchone())

('olist_marketplace',)


In [5]:
files_to_tables = {
    "olist_customers_dataset.csv": "customers",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "product_category_translation"
}

for file_name, table_name in files_to_tables.items():
    file_path = DATA_PATH / file_name
    
    print(f"Importing {file_name} into {table_name}...")
    
    df = pd.read_csv(file_path)
    
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        chunksize=10000,
        method="multi"
    )
    
    print(f"{table_name} imported successfully. Rows: {len(df):,}")

print("All files imported successfully.")

Importing olist_customers_dataset.csv into customers...
customers imported successfully. Rows: 99,441
Importing olist_products_dataset.csv into products...
products imported successfully. Rows: 32,951
Importing olist_sellers_dataset.csv into sellers...
sellers imported successfully. Rows: 3,095
Importing olist_orders_dataset.csv into orders...
orders imported successfully. Rows: 99,441
Importing olist_order_items_dataset.csv into order_items...
order_items imported successfully. Rows: 112,650
Importing olist_order_payments_dataset.csv into order_payments...
order_payments imported successfully. Rows: 103,886
Importing olist_order_reviews_dataset.csv into order_reviews...
order_reviews imported successfully. Rows: 99,224
Importing olist_geolocation_dataset.csv into geolocation...
geolocation imported successfully. Rows: 1,000,163
Importing product_category_name_translation.csv into product_category_translation...
product_category_translation imported successfully. Rows: 71
All files imp

In [6]:
tables = [
    "customers",
    "products",
    "sellers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "geolocation",
    "product_category_translation"
]

for table in tables:
    query = f"SELECT COUNT(*) AS total_rows FROM {table};"
    result = pd.read_sql(query, engine)
    print(f"{table}: {result['total_rows'][0]:,} rows")

customers: 99,441 rows
products: 32,951 rows
sellers: 3,095 rows
orders: 99,441 rows
order_items: 112,650 rows
order_payments: 103,886 rows
order_reviews: 99,224 rows
geolocation: 1,000,163 rows
product_category_translation: 71 rows
